# 🐦 Scraper de Tweets — TwitterAPI.io
Descarga tweets de cuentas específicas usando [twitterapi.io](https://twitterapi.io).
Guarda los resultados en Excel (`.xlsx`) y CSV de respaldo.

> **💡 Anti-gap**: El rango se divide en ventanas pequeñas (`WINDOW_DAYS`) para garantizar cobertura completa.

**Flujo:** Configuración → Funciones → Ejecución → Vista previa

In [ ]:
# ── Librerías ──────────────────────────────────────────────────────────
import requests
import pandas as pd
import time
import json
import traceback
from datetime import datetime, timedelta
from pathlib import Path
from zoneinfo import ZoneInfo

print("✅ Librerías cargadas")

In [ ]:
# ── ⚙️ Configuración (editar aquí) ────────────────────────────────────
API_KEY     = "646ad8f3a4db45409514199060910bc8"  # Tu API key de twitterapi.io
START_DATE  = "2023-10-15"   # Formato: YYYY-MM-DD
END_DATE    = "2023-10-30"
ACCOUNTS    = ["CarlosFGalan"]  # Alcalde de Bogotá
WINDOW_DAYS = 2               # Días por ventana (recomendado: 1-3)
OUTPUT_DIR  = Path(".")        # Carpeta de salida

# Columnas a conservar (None = todas las que devuelve la API)
COLUMNS = [
    "id", "text", "createdAt",
    "likeCount", "retweetCount", "replyCount", "viewCount",
    "lang", "account", "fecha_descarga"
]

print(f"📅 Rango: {START_DATE} → {END_DATE}")
print(f"👤 Cuentas: {ACCOUNTS}")
print(f"🪟 Ventana: {WINDOW_DAYS} días por consulta")

In [ ]:
# ── Funciones ──────────────────────────────────────────────────────────

def date_range_windows(start_str, end_str, window_days):
    """Divide el rango en ventanas de N días para cobertura completa."""
    start = datetime.strptime(start_str, "%Y-%m-%d")
    end   = datetime.strptime(end_str,   "%Y-%m-%d")
    windows = []
    cur = start
    while cur < end:
        nxt = min(cur + timedelta(days=window_days), end)
        windows.append((cur.strftime("%Y-%m-%d"), nxt.strftime("%Y-%m-%d")))
        cur = nxt
    return windows


def fetch_window(query, api_key, max_retries=3):
    """
    Descarga todos los tweets de una ventana de tiempo.
    Incluye detección de loop infinito (cursor repetido o sin tweets nuevos).
    """
    url = "https://api.twitterapi.io/twitter/tweet/advanced_search"
    headers = {"x-api-key": api_key}
    all_tweets, seen_ids = [], set()
    cursor, prev_cursor = None, None

    while True:
        params = {"query": query, "queryType": "Latest"}
        if cursor:
            params["cursor"] = cursor

        for attempt in range(max_retries):
            try:
                r = requests.get(url, headers=headers, params=params, timeout=20)
                r.raise_for_status()
                data = r.json()

                new = [t for t in data.get("tweets", []) if t["id"] not in seen_ids]
                for t in new:
                    seen_ids.add(t["id"])
                all_tweets.extend(new)

                next_cursor = data.get("next_cursor")
                has_next    = data.get("has_next_page", False)

                # Corte de seguridad: cursor repetido o sin tweets nuevos
                if not has_next or next_cursor == prev_cursor or not new:
                    return all_tweets

                prev_cursor = cursor
                cursor      = next_cursor
                break

            except requests.RequestException as e:
                wait = 2 ** attempt
                print(f"    ⚠️ Reintento {attempt+1}: {e} — esperando {wait}s")
                time.sleep(wait)
        else:
            print("    ❌ Error persistente, abortando ventana.")
            break

    return all_tweets


def parse_tweets(tweets, account, columns):
    """Convierte lista de tweets a DataFrame limpio con fechas en hora Colombia."""
    if not tweets:
        return pd.DataFrame()

    df = pd.DataFrame(tweets)

    for col in df.columns:
        df[col] = df[col].apply(
            lambda x: json.dumps(x, ensure_ascii=False)
            if isinstance(x, (dict, list)) else x
        )

    if "createdAt" in df.columns:
        df["createdAt"] = (
            pd.to_datetime(df["createdAt"], errors="coerce", utc=True)
              .dt.tz_convert("America/Bogota")
              .dt.tz_localize(None)
        )

    now = datetime.now(ZoneInfo("America/Bogota"))
    df["account"]        = account
    df["fecha_descarga"] = now.strftime("%Y-%m-%d %H:%M")

    if columns:
        df = df[[c for c in columns if c in df.columns]]

    return df


def download_account_windowed(account, start, end, window_days,
                              api_key, columns):
    """Descarga tweets por ventanas de tiempo para cobertura completa."""
    windows = date_range_windows(start, end, window_days)
    print(f"  🪟 {len(windows)} ventanas de {window_days} días")

    all_raw, seen_global = [], set()

    for i, (w_start, w_end) in enumerate(windows, 1):
        query = f"from:{account} since:{w_start} until:{w_end}"
        print(f"  [{i}/{len(windows)}] {w_start} → {w_end}", end="  ", flush=True)
        try:
            tweets = fetch_window(query, api_key)
            nuevos = [t for t in tweets if t["id"] not in seen_global]
            for t in nuevos:
                seen_global.add(t["id"])
            all_raw.extend(nuevos)
            print(f"+{len(nuevos)} tweets (total: {len(all_raw)})")
        except Exception as e:
            print(f"ERROR: {e}")
            traceback.print_exc()
        time.sleep(0.5)

    df = parse_tweets(all_raw, account, columns)
    if not df.empty:
        df = df.sort_values("createdAt").reset_index(drop=True)
    return df


def save_results(frames, start, end, out_dir):
    """Guarda todos los DataFrames en Excel + CSV de respaldo."""
    combined = pd.concat(frames, ignore_index=True)
    xlsx = out_dir / f"tweets_{start}_to_{end}.xlsx"
    csv  = out_dir / f"tweets_{start}_to_{end}.csv"

    with pd.ExcelWriter(xlsx, engine="openpyxl") as writer:
        combined.to_excel(writer, sheet_name="Todos", index=False)
        for df in frames:
            acc = df["account"].iloc[0]
            df.to_excel(writer, sheet_name=acc[:31], index=False)

    combined.to_csv(csv, index=False, encoding="utf-8-sig", sep=";")
    print(f"\n💾 Excel → {xlsx}")
    print(f"📄 CSV   → {csv}")
    return combined


print("✅ Funciones definidas")

In [ ]:
# ── 🚀 Ejecución ───────────────────────────────────────────────────────
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
frames = []

for i, account in enumerate(ACCOUNTS, 1):
    print(f"\n=== [{i}/{len(ACCOUNTS)}] @{account} ===")
    df = download_account_windowed(
        account, START_DATE, END_DATE, WINDOW_DAYS, API_KEY, COLUMNS
    )
    if not df.empty:
        frames.append(df)
        print(f"  ✅ {len(df)} tweets de @{account}")
    else:
        print(f"  ⚠️ Sin tweets para @{account}")

if frames:
    df_result = save_results(frames, START_DATE, END_DATE, OUTPUT_DIR)
    print(f"🎉 Total: {len(df_result)} tweets de {len(frames)} cuenta(s)")
else:
    print("⚠️ No se descargaron datos.")
    df_result = pd.DataFrame()

In [ ]:
# ── 👀 Vista previa ────────────────────────────────────────────────────
if not df_result.empty:
    print(f"Total tweets : {len(df_result)}")
    print(f"Rango de fechas: {df_result['createdAt'].min()} → {df_result['createdAt'].max()}")
    display(df_result[['createdAt', 'text', 'likeCount', 'retweetCount', 'viewCount']].head(10))